### Import

In [ ]:
import os
import pandas as pd
pd.options.display.float_format = '{:.3f}'.format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
import numpy as np
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB
from itertools import product
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from tqdm import tqdm
from functions_utils import *
from functions_data import *
from functions_optimize import *
from functions_eval import *

generation_data, I, T = load_generation_data(date_filter="2022-07-18")
S, R, P_RT, K, K0, M1, M2 = load_parameters(I, T, generation_data)
P_DA, P_PN = load_price_data()

In [ ]:
def save_holistic_results(x_hol, a_hol, yp_hol, ym_hol, z_hol, zc_hol, zd_hol, 
                               ep_hol, bp_hol, em_hol, bm_hol, d_hol, dp_hol, dm_hol, 
                               obj_hol, I, T, S, base_dir="opt_result"):
    """
    최적화 결과를 변수별 CSV 파일로 저장
    
    Parameters:
    - 모든 최적화 변수들
    - I: 발전소 개수
    - T: 시간 구간 개수  
    - S: 시나리오 개수
    - base_dir: 기본 저장 디렉토리 (기본값: "opt_result")
    """
    
    # 폴더 구조 생성: opt_result/i_{I}_s_{S}/
    save_dir = os.path.join(base_dir, f"i_{I}_s_{S}")
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    print(f"🔄 최적화 결과 저장 중... (폴더: {save_dir})")
    
    # 1. x_hol: 발전량 (I × T)
    x_data = []
    for i in range(I):
        for t in range(T):
            x_data.append({
                'i': i,
                't': t,
                'x_hol': x_hol[i, t]
            })
    pd.DataFrame(x_data).to_csv(f"{save_dir}/x_hol.csv", index=False)
    print("✅ x_hol.csv 저장 완료")
    
    # 2. a_hol: 총 발전량 (T,)
    a_data = []
    for t in range(T):
        a_data.append({
            't': t,
            'a_hol': a_hol[t]
        })
    pd.DataFrame(a_data).to_csv(f"{save_dir}/a_hol.csv", index=False)
    print("✅ a_hol.csv 저장 완료")
    
    # 3. yp_hol: 양의 편차 (I × T × S)
    yp_data = []
    for i in range(I):
        for t in range(T):
            for s in range(S):
                yp_data.append({
                    'i': i,
                    't': t,
                    's': s,
                    'yp_hol': yp_hol[i, t, s]
                })
    pd.DataFrame(yp_data).to_csv(f"{save_dir}/yp_hol.csv", index=False)
    print("✅ yp_hol.csv 저장 완료")
    
    # 4. ym_hol: 음의 편차 (I × T × S)
    ym_data = []
    for i in range(I):
        for t in range(T):
            for s in range(S):
                ym_data.append({
                    'i': i,
                    't': t,
                    's': s,
                    'ym_hol': ym_hol[i, t, s]
                })
    pd.DataFrame(ym_data).to_csv(f"{save_dir}/ym_hol.csv", index=False)
    print("✅ ym_hol.csv 저장 완료")
    
    # 5. z_hol: 배터리 상태 (I × T × S)
    z_data = []
    for i in range(I):
        for t in range(T):
            for s in range(S):
                z_data.append({
                    'i': i,
                    't': t,
                    's': s,
                    'z_hol': z_hol[i, t, s]
                })
    pd.DataFrame(z_data).to_csv(f"{save_dir}/z_hol.csv", index=False)
    print("✅ z_hol.csv 저장 완료")
    
    # 6. zc_hol: 배터리 충전 (I × T × S)
    zc_data = []
    for i in range(I):
        for t in range(T):
            for s in range(S):
                zc_data.append({
                    'i': i,
                    't': t,
                    's': s,
                    'zc_hol': zc_hol[i, t, s]
                })
    pd.DataFrame(zc_data).to_csv(f"{save_dir}/zc_hol.csv", index=False)
    print("✅ zc_hol.csv 저장 완료")
    
    # 7. zd_hol: 배터리 방전 (I × T × S)
    zd_data = []
    for i in range(I):
        for t in range(T):
            for s in range(S):
                zd_data.append({
                    'i': i,
                    't': t,
                    's': s,
                    'zd_hol': zd_hol[i, t, s]
                })
    pd.DataFrame(zd_data).to_csv(f"{save_dir}/zd_hol.csv", index=False)
    print("✅ zd_hol.csv 저장 완료")
    
    # 8. ep_hol: 에너지 양의 편차 (I × T × S)
    ep_data = []
    for i in range(I):
        for t in range(T):
            for s in range(S):
                ep_data.append({
                    'i': i,
                    't': t,
                    's': s,
                    'ep_hol': ep_hol[i, t, s]
                })
    pd.DataFrame(ep_data).to_csv(f"{save_dir}/ep_hol.csv", index=False)
    print("✅ ep_hol.csv 저장 완료")
    
    # 9. bp_hol: 총 에너지 양의 편차 (T × S)
    bp_data = []
    for t in range(T):
        for s in range(S):
            bp_data.append({
                't': t,
                's': s,
                'bp_hol': bp_hol[t, s]
            })
    pd.DataFrame(bp_data).to_csv(f"{save_dir}/bp_hol.csv", index=False)
    print("✅ bp_hol.csv 저장 완료")
    
    # 10. em_hol: 에너지 음의 편차 (I × T × S)
    em_data = []
    for i in range(I):
        for t in range(T):
            for s in range(S):
                em_data.append({
                    'i': i,
                    't': t,
                    's': s,
                    'em_hol': em_hol[i, t, s]
                })
    pd.DataFrame(em_data).to_csv(f"{save_dir}/em_hol.csv", index=False)
    print("✅ em_hol.csv 저장 완료")
    
    # 11. bm_hol: 총 에너지 음의 편차 (T × S)
    bm_data = []
    for t in range(T):
        for s in range(S):
            bm_data.append({
                't': t,
                's': s,
                'bm_hol': bm_hol[t, s]
            })
    pd.DataFrame(bm_data).to_csv(f"{save_dir}/bm_hol.csv", index=False)
    print("✅ bm_hol.csv 저장 완료")
    
    # 12. d_hol: 교환량 (I × I × T × S) - i와 j로 구분
    d_data = []
    for i in range(I):
        for j in range(I):
            for t in range(T):
                for s in range(S):
                    if i != j:  # 자기 자신과의 교환은 제외 (항상 0)
                        d_data.append({
                            'i': i,
                            'j': j,
                            't': t,
                            's': s,
                            'd_hol': d_hol[i, j, t, s]
                        })
    pd.DataFrame(d_data).to_csv(f"{save_dir}/d_hol.csv", index=False)
    print("✅ d_hol.csv 저장 완료")
    
    # 13. dp_hol: 발전소별 총 송출량 (I × T × S)
    dp_data = []
    for i in range(I):
        for t in range(T):
            for s in range(S):
                dp_data.append({
                    'i': i,
                    't': t,
                    's': s,
                    'dp_hol': dp_hol[i, t, s]
                })
    pd.DataFrame(dp_data).to_csv(f"{save_dir}/dp_hol.csv", index=False)
    print("✅ dp_hol.csv 저장 완료")
    
    # 14. dm_hol: 발전소별 총 수신량 (I × T × S)
    dm_data = []
    for i in range(I):
        for t in range(T):
            for s in range(S):
                dm_data.append({
                    'i': i,
                    't': t,
                    's': s,
                    'dm_hol': dm_hol[i, t, s]
                })
    pd.DataFrame(dm_data).to_csv(f"{save_dir}/dm_hol.csv", index=False)
    print("✅ dm_hol.csv 저장 완료")
    
    # 15. obj_hol: 목적함수 값 (스칼라)
    obj_data = [{
        'obj_hol': obj_hol,
        'I': I,
        'T': T,
        'S': S
    }]
    pd.DataFrame(obj_data).to_csv(f"{save_dir}/obj_hol.csv", index=False)
    print("✅ obj_hol.csv 저장 완료")
    
    print(f"\n🎉 모든 변수가 '{save_dir}' 폴더에 저장되었습니다!")
    print(f"📁 총 {15}개 파일 생성")
    
    return save_dir

def load_holistic_results(I, T, S, base_dir="opt_result"):
    """
    저장된 CSV 파일들로부터 최적화 변수들을 불러오기
    """
    save_dir = os.path.join(base_dir, f"i_{I}_s_{S}")
    print(f"🔄 최적화 결과 불러오는 중... (폴더: {save_dir})")
    
    results = {}
    
    # 1. x_hol 불러오기 (I × T)
    x_df = pd.read_csv(f"{save_dir}/x_hol.csv")
    x_hol = np.zeros((I, T))
    for _, row in x_df.iterrows():
        x_hol[int(row['i']), int(row['t'])] = row['x_hol']
    results['x_hol'] = x_hol
    print("✅ x_hol 불러오기 완료")
    
    # 2. a_hol 불러오기 (T,)
    a_df = pd.read_csv(f"{save_dir}/a_hol.csv")
    a_hol = np.zeros(T)
    for _, row in a_df.iterrows():
        a_hol[int(row['t'])] = row['a_hol']
    results['a_hol'] = a_hol
    print("✅ a_hol 불러오기 완료")
    
    # 3. yp_hol 불러오기 (I × T × S)
    yp_df = pd.read_csv(f"{save_dir}/yp_hol.csv")
    yp_hol = np.zeros((I, T, S))
    for _, row in yp_df.iterrows():
        yp_hol[int(row['i']), int(row['t']), int(row['s'])] = row['yp_hol']
    results['yp_hol'] = yp_hol
    print("✅ yp_hol 불러오기 완료")
    
    # 4. ym_hol 불러오기 (I × T × S)
    ym_df = pd.read_csv(f"{save_dir}/ym_hol.csv")
    ym_hol = np.zeros((I, T, S))
    for _, row in ym_df.iterrows():
        ym_hol[int(row['i']), int(row['t']), int(row['s'])] = row['ym_hol']
    results['ym_hol'] = ym_hol
    print("✅ ym_hol 불러오기 완료")
    
    # 5. z_hol 불러오기 (I × T × S)
    z_df = pd.read_csv(f"{save_dir}/z_hol.csv")
    z_hol = np.zeros((I, T, S))
    for _, row in z_df.iterrows():
        z_hol[int(row['i']), int(row['t']), int(row['s'])] = row['z_hol']
    results['z_hol'] = z_hol
    print("✅ z_hol 불러오기 완료")
    
    # 6. zc_hol 불러오기 (I × T × S)
    zc_df = pd.read_csv(f"{save_dir}/zc_hol.csv")
    zc_hol = np.zeros((I, T, S))
    for _, row in zc_df.iterrows():
        zc_hol[int(row['i']), int(row['t']), int(row['s'])] = row['zc_hol']
    results['zc_hol'] = zc_hol
    print("✅ zc_hol 불러오기 완료")
    
    # 7. zd_hol 불러오기 (I × T × S)
    zd_df = pd.read_csv(f"{save_dir}/zd_hol.csv")
    zd_hol = np.zeros((I, T, S))
    for _, row in zd_df.iterrows():
        zd_hol[int(row['i']), int(row['t']), int(row['s'])] = row['zd_hol']
    results['zd_hol'] = zd_hol
    print("✅ zd_hol 불러오기 완료")
    
    # 8. ep_hol 불러오기 (I × T × S)
    ep_df = pd.read_csv(f"{save_dir}/ep_hol.csv")
    ep_hol = np.zeros((I, T, S))
    for _, row in ep_df.iterrows():
        ep_hol[int(row['i']), int(row['t']), int(row['s'])] = row['ep_hol']
    results['ep_hol'] = ep_hol
    print("✅ ep_hol 불러오기 완료")
    
    # 9. bp_hol 불러오기 (T × S)
    bp_df = pd.read_csv(f"{save_dir}/bp_hol.csv")
    bp_hol = np.zeros((T, S))
    for _, row in bp_df.iterrows():
        bp_hol[int(row['t']), int(row['s'])] = row['bp_hol']
    results['bp_hol'] = bp_hol
    print("✅ bp_hol 불러오기 완료")
    
    # 10. em_hol 불러오기 (I × T × S)
    em_df = pd.read_csv(f"{save_dir}/em_hol.csv")
    em_hol = np.zeros((I, T, S))
    for _, row in em_df.iterrows():
        em_hol[int(row['i']), int(row['t']), int(row['s'])] = row['em_hol']
    results['em_hol'] = em_hol
    print("✅ em_hol 불러오기 완료")
    
    # 11. bm_hol 불러오기 (T × S)
    bm_df = pd.read_csv(f"{save_dir}/bm_hol.csv")
    bm_hol = np.zeros((T, S))
    for _, row in bm_df.iterrows():
        bm_hol[int(row['t']), int(row['s'])] = row['bm_hol']
    results['bm_hol'] = bm_hol
    print("✅ bm_hol 불러오기 완료")
    
    # 12. d_hol 불러오기 (I × I × T × S)
    d_df = pd.read_csv(f"{save_dir}/d_hol.csv")
    d_hol = np.zeros((I, I, T, S))
    for _, row in d_df.iterrows():
        d_hol[int(row['i']), int(row['j']), int(row['t']), int(row['s'])] = row['d_hol']
    results['d_hol'] = d_hol
    print("✅ d_hol 불러오기 완료")
    
    # 13. dp_hol 불러오기 (I × T × S)
    dp_df = pd.read_csv(f"{save_dir}/dp_hol.csv")
    dp_hol = np.zeros((I, T, S))
    for _, row in dp_df.iterrows():
        dp_hol[int(row['i']), int(row['t']), int(row['s'])] = row['dp_hol']
    results['dp_hol'] = dp_hol
    print("✅ dp_hol 불러오기 완료")
    
    # 14. dm_hol 불러오기 (I × T × S)
    dm_df = pd.read_csv(f"{save_dir}/dm_hol.csv")
    dm_hol = np.zeros((I, T, S))
    for _, row in dm_df.iterrows():
        dm_hol[int(row['i']), int(row['t']), int(row['s'])] = row['dm_hol']
    results['dm_hol'] = dm_hol
    print("✅ dm_hol 불러오기 완료")
    
    # 15. obj_hol 불러오기 (스칼라)
    obj_df = pd.read_csv(f"{save_dir}/obj_hol.csv")
    obj_hol = obj_df['obj_hol'].iloc[0]
    results['obj_hol'] = obj_hol
    print("✅ obj_hol 불러오기 완료")
    
    print(f"\n🎉 모든 변수 불러오기 완료!")
    return results

In [ ]:
if __name__ == "__main__":
    print("[Holistic Aggregation Model optimization]")
    
    # 최적화 실행
    x_hol, a_hol, yp_hol, ym_hol, z_hol, zc_hol, zd_hol, ep_hol, bp_hol, em_hol, bm_hol, d_hol, dp_hol, dm_hol, obj_hol = optimize_hol(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1, M2)
    print("-" * 100)
    
    # 결과 저장 - opt_result/i_10_s_200/ 폴더에 저장됨
    save_dir = save_holistic_results(
        x_hol, a_hol, yp_hol, ym_hol, z_hol, zc_hol, zd_hol, 
        ep_hol, bp_hol, em_hol, bm_hol, d_hol, dp_hol, dm_hol, 
        obj_hol, I, T, S
    )
    
    # 불러오기
    loaded_results = load_holistic_results(I, T, S)
    x_hol_loaded = loaded_results['x_hol']
    obj_hol_loaded = loaded_results['obj_hol']